# Churn Analysis — Tidemill API vs Stripe

Compare logo churn and revenue churn from the Tidemill engine with Stripe subscription data.

In [1]:
import os
import warnings
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import requests
import stripe

warnings.filterwarnings("ignore", "Unverified HTTPS")

API = os.environ.get("TIDEMILL_API", "http://localhost:8000")
stripe.api_key = os.environ["STRIPE_API_KEY"]

START, END = "2025-09-01", "2026-03-31"
plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True, "grid.alpha": 0.3})


def api_get(path, **params):
    r = requests.get(f"{API}{path}", params=params)
    r.raise_for_status()
    return r.json()

## 1. Logo Churn from Tidemill API

In [2]:
churn_data = api_get("/api/metrics/churn", start=START, end=END, type="logo")
print("Logo churn response:", churn_data)
print()

# Also fetch via the generic POST endpoint for more detail
churn_detail = api_get("/api/metrics/churn", start=START, end=END, type="revenue")
print("Revenue churn response:", churn_detail)

Logo churn response: None

Revenue churn response: None


## 2. Churn from Stripe (ground truth)

Count canceled subscriptions and lost MRR directly from Stripe.

In [3]:
def stripe_mrr_for_sub(sub):
    total = 0
    for item in sub["items"]["data"]:
        price = item["price"]
        qty = item.get("quantity", 1) or 1
        unit_amount = price.get("unit_amount", 0) or 0
        amount = unit_amount * qty
        rec = price.get("recurring") or {}
        interval = rec.get("interval", "month")
        interval_count = rec.get("interval_count", 1) or 1
        if interval == "month":
            total += amount // interval_count
        elif interval == "year":
            total += amount // (12 * interval_count)
    return total


# Collect all subscriptions across test clocks
clock_ids = [c.id for c in stripe.test_helpers.TestClock.list(limit=100).auto_paging_iter()]
all_subs = []
for cid in clock_ids:
    for sub in stripe.Subscription.list(
        limit=100, test_clock=cid, status="all"
    ).auto_paging_iter():
        all_subs.append(dict(sub))

df = pd.DataFrame(
    [
        {
            "id": s["id"],
            "customer": s["customer"],
            "status": s["status"],
            "mrr_cents": stripe_mrr_for_sub(s),
            "canceled_at": datetime.fromtimestamp(s["canceled_at"], tz=UTC).strftime("%Y-%m-%d")
            if s.get("canceled_at")
            else None,
            "cancel_at_period_end": s.get("cancel_at_period_end", False),
        }
        for s in all_subs
    ]
)

canceled = df[df.status == "canceled"]
active = df[df.status == "active"]

total_subs = len(df)
churned_count = len(canceled)
active_count = len(active)

print(f"Total subscriptions:  {total_subs}")
print(f"Active:               {active_count}")
print(f"Canceled (churned):   {churned_count}")
print(f"Logo churn rate:      {churned_count / total_subs * 100:.1f}%")
print()
print("Churned subscriptions:")
canceled[["id", "customer", "canceled_at", "mrr_cents"]]

Total subscriptions:  15
Active:               12
Canceled (churned):   3
Logo churn rate:      20.0%

Churned subscriptions:


,id,customer,canceled_at,mrr_cents
0,sub_1TIOYYCMLOTbZAd7NQotGchs,cus_UGwRg3dZWjxkgA,2025-09-01,2
6,sub_1TIOYECMLOTbZAd7WsM4EszF,cus_UGwR1nMQ3yAdsS,2025-10-01,7900
7,sub_1TIOYBCMLOTbZAd79hPHNsjw,cus_UGwR2i9czhY6o8,2025-10-01,2


## 3. Churn visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Logo churn: active vs churned
labels = ["Active", "Churned"]
sizes = [active_count, churned_count]
axes[0].pie(
    sizes,
    labels=labels,
    autopct="%1.0f%%",
    colors=["#2ecc71", "#e74c3c"],
    startangle=90,
    explode=[0, 0.05],
)
axes[0].set_title(
    f"Logo Churn: {churned_count}/{total_subs} ({churned_count / total_subs * 100:.0f}%)"
)

# Revenue churn: MRR retained vs lost
active_mrr = active.mrr_cents.sum()
lost_mrr = canceled.mrr_cents.sum()  # MRR at time of cancel (now 0)
# Use MRR movements from Tidemill for lost MRR
breakdown = api_get("/api/metrics/mrr/breakdown", start=START, end=END)
churn_movement = [b for b in breakdown if b["movement_type"] == "churn"]
lost_from_api = abs(int(churn_movement[0]["amount_base"])) if churn_movement else 0

axes[1].bar(
    ["Active MRR", "Churned MRR (lost)"],
    [active_mrr / 100, lost_from_api / 100],
    color=["#2ecc71", "#e74c3c"],
)
axes[1].set_title("Revenue Impact of Churn")
axes[1].set_ylabel("MRR ($)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

for i, v in enumerate([active_mrr / 100, lost_from_api / 100]):
    axes[1].text(i, v + 10, f"${v:,.0f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

if lost_from_api > 0:
    print(f"Revenue churn rate: {lost_from_api / (active_mrr + lost_from_api) * 100:.1f}%")

## 4. Churn events from Tidemill vs Stripe cancellations

In [ ]:
# Compare: Stripe canceled subs vs Tidemill churn events
stripe_canceled_ids = set(canceled["id"])
churn_events = [b for b in breakdown if b["movement_type"] == "churn"]

print(f"Stripe canceled subscriptions:  {len(stripe_canceled_ids)}")
print(f"Tidemill churn events (MRR):    {len(churn_events)}")
print()
print("MRR movement summary from Tidemill:")
for b in breakdown:
    amt = int(b["amount_base"]) / 100
    print(f"  {b['movement_type']:15s}  ${amt:>10,.2f}")

## 5. Monthly Churn Rate Timeline

Compute logo and revenue churn rates per month to visualize churn trends over time.

In [ ]:
# Monthly churn rates from Tidemill API
months = pd.date_range(START, END, freq="MS")
monthly_churn = []
for i in range(len(months) - 1):
    s = months[i].strftime("%Y-%m-%d")
    e = months[i + 1].strftime("%Y-%m-%d")
    logo = api_get("/api/metrics/churn", start=s, end=e, type="logo")
    revenue = api_get("/api/metrics/churn", start=s, end=e, type="revenue")
    monthly_churn.append(
        {
            "month": months[i].strftime("%Y-%m"),
            "logo_churn": logo,
            "revenue_churn": revenue,
        }
    )

df_mc = pd.DataFrame(monthly_churn)
print("Monthly churn rates:")
for _, row in df_mc.iterrows():
    logo = f"{row.logo_churn * 100:.1f}%" if row.logo_churn is not None else "N/A"
    rev = f"{row.revenue_churn * 100:.1f}%" if row.revenue_churn is not None else "N/A"
    print(f"  {row.month}:  logo={logo:>6s}   revenue={rev:>6s}")

In [ ]:
# Plot monthly churn rates
fig, ax = plt.subplots(figsize=(12, 5))

logo_vals = [r if r is not None else float("nan") for r in df_mc.logo_churn]
rev_vals = [r if r is not None else float("nan") for r in df_mc.revenue_churn]

x = range(len(df_mc))
ax.plot(
    x,
    [v * 100 for v in logo_vals],
    "o-",
    color="#e74c3c",
    linewidth=2,
    markersize=8,
    label="Logo Churn",
)
ax.plot(
    x,
    [v * 100 for v in rev_vals],
    "s--",
    color="#e67e22",
    linewidth=2,
    markersize=8,
    label="Revenue Churn",
)

for i, (lv, rv) in enumerate(zip(logo_vals, rev_vals, strict=True)):
    if not pd.isna(lv):
        ax.annotate(
            f"{lv * 100:.1f}%",
            (i, lv * 100),
            textcoords="offset points",
            xytext=(0, 12),
            ha="center",
            fontweight="bold",
            fontsize=9,
            color="#e74c3c",
        )
    if not pd.isna(rv):
        ax.annotate(
            f"{rv * 100:.1f}%",
            (i, rv * 100),
            textcoords="offset points",
            xytext=(0, -16),
            ha="center",
            fontweight="bold",
            fontsize=9,
            color="#e67e22",
        )

ax.set_xticks(x)
ax.set_xticklabels(df_mc.month, rotation=45)
ax.set_title("Monthly Churn Rate")
ax.set_ylabel("Churn Rate (%)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.legend()
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## 6. Monthly Churned MRR

Lost MRR per month from the MRR waterfall — shows the dollar impact of churn over time.

In [ ]:
# Monthly churned MRR from waterfall
waterfall = api_get("/api/metrics/mrr/waterfall", start=START, end=END)
df_wf = pd.DataFrame(waterfall)
df_wf["churn_dollars"] = df_wf["churn"].apply(lambda c: abs(c) / 100)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(df_wf.month, df_wf.churn_dollars, color="#e74c3c", alpha=0.8)

for bar, val in zip(bars, df_wf.churn_dollars, strict=True):
    if val > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 2,
            f"${val:,.0f}",
            ha="center",
            fontweight="bold",
            fontsize=10,
        )

ax.set_title("Monthly Churned MRR")
ax.set_ylabel("Lost MRR ($)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.set_xticklabels(df_wf.month, rotation=45)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

print(f"\nTotal churned MRR: ${df_wf.churn_dollars.sum():,.2f}")